# 🗺️ TavilyMap & TavilyExtract Tutorial

> **📚 Part of the LangChain Course: Building AI Agents & RAG Apps**  
> [🎓 Get the full course](https://www.udemy.com/course/langchain/?referralCode=D981B8213164A3EA91AC)


This notebook demonstrates two powerful tools from Tavily AI:
- **TavilyMap**: Automatically discovers and maps website structures
- **TavilyExtract**: Extracts clean, structured content from web pages

Perfect for documentation scraping, research, and content extraction! 🚀

---


## 📦 Setup & Installation

First, let's install the required packages and set up our environment.


In [1]:
# Install required packages
!pip install langchain-tavily certifi

# For pretty printing and visualization
!pip install rich pandas



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import asyncio
import os
import ssl
from typing import Any, Dict, List

import certifi
from langchain_tavily import TavilyExtract, TavilyMap
from rich.console import Console
from rich.panel import Panel

# Configure SSL context
ssl_context = ssl.create_default_context(cafile=certifi.where())
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

# Initialize rich console for pretty printing
console = Console()


print("✅ All imports successful!")


✅ All imports successful!


## 🔑 API Key Setup

You'll need a Tavily API key to use these tools. Get yours at [tavily.com](https://app.tavily.com/home).

Set environment variable `TAVILY_API_KEY`


In [3]:


# Set directly (uncomment and add your key)
# tavily_api_key = "your_tavily_api_key_here"

os.environ["TAVILY_API_KEY"] = "tvly-1JX5gf-jD79SgF03J3b4Bpu5yj01uMz2GcKHCqkfSjr7xhohk"

## 🗺️ TavilyMap: Website Structure Discovery

TavilyMap automatically discovers and maps website structures by crawling through links. It's perfect for:
- Documentation sites
- Blog archives
- Knowledge bases
- Any structured website

### Key Parameters:
- `max_depth`: How deep to crawl (default: 3)
- `max_breadth`: How many links per page (default: 10)
- `max_pages`: Maximum total pages to discover (default: 100)


In [4]:
# Initialize TavilyMap with custom settings
tavily_map = TavilyMap(
    max_depth=3,        # Crawl up to 3 levels deep
    max_breadth=15,     # Follow up to 15 links per page
    max_pages=50        # Limit to 50 total pages for demo
)

print("✅ TavilyMap initialized successfully!")


✅ TavilyMap initialized successfully!


### 📊 Demo: Mapping a Documentation Site

Let's map the structure of a popular documentation site. We'll use the FastAPI documentation as an example.


In [5]:
# Example website to map
demo_url = "https://python.langchain.com/docs/introduction/"

console.print(f"🔍 Mapping website structure for: {demo_url}", style="bold blue")
console.print("This may take a moment...")

# Map the website structure
site_map = tavily_map.invoke(demo_url)

# Display results
urls = site_map.get('results', [])
console.print(f"\n✅ Successfully mapped {len(urls)} URLs!", style="bold green")

# Show first 10 URLs as examples
console.print("\n📋 First 50 discovered URLs:", style="bold yellow")
for i, url in enumerate(urls[:50], 1):
    console.print(f"  {i:2d}. {url}")

if len(urls) > 10:
    console.print(f"  ... and {len(urls) - 50} more URLs")


🔍 Mapping website structure for: https://python.langchain.com/docs/introduction/

This may take a moment...

✅ Successfully mapped 26 URLs!

📋 First 50 discovered URLs:

1. https://python.langchain.com/docs/introduction

2. https://python.langchain.com/use-these-docs

... (21 outputs left out)

24. https://python.langchain.com/oss/python/langchain/tools

25. https://python.langchain.com/oss/python/langchain/guardrails

26. https://python.langchain.com/oss/python/langchain/deploy

... and -24 more URLs

## 🔍 TavilyExtract: Clean Content Extraction

TavilyExtract takes URLs and returns clean, structured content without ads, navigation, or other noise. Perfect for:
- Documentation processing
- Content analysis
- Research and data collection
- Building knowledge bases

### Key Features:
- Removes HTML markup and navigation
- Extracts main content only
- Handles JavaScript-rendered content
- Batch processing support

In [6]:
# Initialize TavilyExtract
tavily_extract = TavilyExtract()

print("✅ TavilyExtract initialized successfully!")


✅ TavilyExtract initialized successfully!


### 📄 Demo: Extracting Content from URLs

Let's extract clean content from some of the URLs we discovered earlier.


In [7]:
# Select a few interesting URLs for extraction
sample_urls = [urls[15]]  # Take first 5 URLs
console.print(f"📚 Extracting content from {len(sample_urls)} URLs...", style="bold blue")

# Extract content
extraction_result = await tavily_extract.ainvoke(input={"urls": sample_urls})

# Display results
extracted_docs = extraction_result.get('results', [])
console.print(f"\n✅ Successfully extracted {len(extracted_docs)} documents!", style="bold green")

# Show summary of each extracted document
for i, doc in enumerate(extracted_docs, 1):
    url = doc.get('url', 'Unknown')
    content = doc.get('raw_content', '')

    # Create a panel for each document
    panel_content = f"""URL: {url}
Content Length: {len(content):,} characters
Preview: {content}..."""

    console.print(Panel(panel_content, title=f"Document {i}", border_style="blue"))
    print()  # Add spacing


📚 Extracting content from 1 URLs...

✅ Successfully extracted 1 documents!

╭────────────────────────────────────────────────── Document 1 ───────────────────────────────────────────────────╮
│ URL: https://python.langchain.com/oss/python/langchain/overview.md                                              │
│ Content Length: 402,995 characters                                                                              │
│ Preview: > ## Documentation Index                                                                               │
│ >                                                                                                               │
│ > Fetch the complete documentation index at:</llms.txt>                                                         │
│ >                                                                                                               │
│ > Use this file to discover all available pages before exploring further.                                       │
│                                                                                                                 │
│ Would you notice if your agent got worse in production? [Tell us in our annual                                  │
│ survey](https://form.typeform.com/to/U2EAO1Lz) for the chance to win LangChain swag.                            │
│                                                                                                                 │
│ [Docs by LangChain home page](/)                                                                                │
│                                                                                                                 │
│ [Overview](/build-overview)[Managed Deep Agents](/langsmith/python/managed-deep-agents-overview)[Deep           │
│ Agents](/oss/python/deepagents/overview)[LangChain](/oss/python/langchain/overview)[LangGraph](/oss/python/lang │
│ graph/overview)[OpenWiki](/oss/openwiki/overview)[Integrations](/oss/python/integrations/providers/overview)[Le │
│ arn](/oss/python/learn)[Reference](/oss/python/reference/overview)[Contribute](/oss/python/contributing/overvie │
│ w)                                                                                                              │
│                                                                                                                 │
│ # LangChain overview                                                                                            │
│                                                                                                                 │
│ LangChain provides create\_agent: a minimal, highly configurable agent harness. Compose exactly the agent your  │
│ use case needs from model, tools, prompt, and middleware.                                                       │
│                                                                                                                 │
│ **Agent = Model + Harness.** LangChain provides `create_agent`: a minimal, highly configurable harness. The     │
│ harness is everything around the model loop: the prompt, the tools, and any middleware that shapes behavior.    │
│ Start with the primitives and compose exactly what your use case needs. Supports [OpenAI, Anthropic, Google,    │
│ and more](/oss/python/integrations/providers/overview).                                                         │
│                                                                                                                 │
│ **LangChain vs. LangGraph vs. Deep Agents**Start with [Managed Deep                                             │
│ Agents](/langsmith/python/managed-deep-agents-overview) for a batteries-included harness (context compression,  │
│ virtual filesystem, subagents) when you want to run production agents without having to manage infrastructure.  │
│ Alternatively, for maximum flexibility while using the same Deep Agents harness, use the [Deep Agents           │
│ SDK](/oss/python/deepagents/overview#self-host-with-th

### ⚡ Batch Processing Demo

For larger datasets, we can process URLs in batches to optimize performance and handle rate limits.


In [8]:
def chunk_urls(urls: List[str], chunk_size: int = 3) -> List[List[str]]:
    """Split URLs into chunks of specified size."""
    chunks = []
    for i in range(0, len(urls), chunk_size):
        chunk = urls[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

async def extract_batch(urls: List[str], batch_num: int) -> List[Dict[str, Any]]:
    """Extract documents from a batch of URLs."""
    try:
        console.print(f"🔄 Processing batch {batch_num} with {len(urls)} URLs", style="blue")
        docs = await tavily_extract.ainvoke(input={"urls": urls})
        results = docs.get('results', [])
        console.print(f"✅ Batch {batch_num} completed - extracted {len(results)} documents", style="green")
        return results
    except Exception as e:
        console.print(f"❌ Batch {batch_num} failed: {e}", style="red")
        return []

# Process a larger set of URLs in batches
url_batches = chunk_urls(urls[:9], chunk_size=3) # Take first 9 URLs for batch demo, split to batches of 3

console.print(f"📦 Processing 9 URLs in {len(url_batches)} batches", style="bold yellow")

# Process batches concurrently
tasks = [extract_batch(batch, i + 1) for i, batch in enumerate(url_batches)]
batch_results = await asyncio.gather(*tasks)

# Flatten results
all_extracted = []
doc_counter = 1


# Remember, batch_results contains a list of results for each batch.
# batch_results = [
#     [{"url": "url1"}, {"url": "url2"}, {"url": "url3"}],
#     [{"url": "url4"}, {"url": "url5"}, {"url": "url6"}],
#     [{"url": "url7"}, {"url": "url8"}, {"url": "url9"}]
# ]

for batch_result in batch_results:
    all_extracted.extend(batch_result)
    for doc in batch_result:
        # all_extracted.append(doc)

        url = doc.get("url", "Unknown")
        content = doc.get("raw_content", "")

        panel_content = f"""URL: {url}
Content Length: {len(content):,} characters
Preview: {content[:1000]}..."""

        console.print(
            Panel(
                panel_content,
                title=f"Document {doc_counter}",
                border_style="blue"
            )
        )

        doc_counter += 1

console.print(f"\n🎉 Batch processing complete! Total documents extracted: {len(all_extracted)}", style="bold green")


📦 Processing 9 URLs in 3 batches

🔄 Processing batch 1 with 3 URLs

🔄 Processing batch 2 with 3 URLs

🔄 Processing batch 3 with 3 URLs

✅ Batch 1 completed - extracted 3 documents

✅ Batch 2 completed - extracted 3 documents

... (7 outputs left out)

╭────────────────────────────────────────────────── Document 7 ───────────────────────────────────────────────────╮
│ URL: https://python.langchain.com/langsmith/trace-with-langchain                                                │
│ Content Length: 389,176 characters                                                                              │
│ Preview: > ## Documentation Index                                                                               │
│ >                                                                                                               │
│ > Fetch the complete documentation index at:</llms.txt>                                                         │
│ >                                                                                                               │
│ > Use this file to discover all available pages before exploring further.                                       │
│                                                                                                                 │
│ Would you notice if your agent got worse in production? [Tell us in our annual                                  │
│ survey](https://form.typeform.com/to/U2EAO1Lz) for the chance to win LangChain swag.                            │
│                                                                                                                 │
│ [Docs by LangChain home page](/)                                                                                │
│                                                                                                                 │
│ [Overview](/build-overview)[Managed Deep Agents](/langsmith/python/managed-deep-agents-overview)[Deep           │
│ Agents](/oss/python/deepagents/overview)[LangChain](/oss/python/langchain/overview)[LangGraph](/oss/python/lang │
│ graph/overview)[OpenWiki](/oss/openwiki/overview)[Integrations](/oss/python/integrations/providers/overview)[Le │
│ arn](/oss/python/learn)[Reference](/oss/python/reference/overview)[Contribute](/oss/python/contributing/overvie │
│ w)                                                                                                              │
│                                                                                                                 │
│ # LangChain overview                                                                                            │
│                                                                                                                 │
│ LangChain provides create\_agent: a minimal, highly configurable agent harness. Compose exactly the agent your  │
│ use case needs from model, tools, prompt, and middleware.                                                       │
│                                                                                                                 │
│ **Agent = ...                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Document 8 ───────────────────────────────────────────────────╮
│ URL: https://python.langchain.com/langsmith/engine                                                              │
│ Content Length: 389,176 characters                                                                              │
│ Preview: > ## Documentation Index                                                                               │
│ >                                                                                                               │
│ > Fetch the complete documentation index at:</llms.txt>                                                         │
│ >                                                                                                               │
│ > Use this file to discover all available pages before exploring further.                                       │
│                                                                                                                 │
│ Would you notice if your agent got worse in production? [Tell us in our annual                                  │
│ survey](https://form.typeform.com/to/U2EAO1Lz) for the chance to win LangChain swag.                            │
│                                                                                                                 │
│ [Docs by LangChain home page](/)                                                                                │
│                                                                                                                 │
│ [Overview](/build-overview)[Managed Deep Agents](/langsmith/python/managed-deep-agents-overview)[Deep           │
│ Agents](/oss/python/deepagents/overview)[LangChain](/oss/python/langchain/overview)[LangGraph](/oss/python/lang │
│ graph/overview)[OpenWiki](/oss/openwiki/overview)[Integrations](/oss/python/integrations/providers/overview)[Le │
│ arn](/oss/python/learn)[Reference](/oss/python/reference/overview)[Contribute](/oss/python/contributing/overvie │
│ w)                                                                                                              │
│                                                                                                                 │
│ # LangChain overview                                                                                            │
│                                                                                                                 │
│ LangChain provides create\_agent: a minimal, highly configurable agent harness. Compose exactly the agent your  │
│ use case needs from model, tools, prompt, and middleware.                                                       │
│                                                                                                                 │
│ **Agent = ...                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Document 9 ───────────────────────────────────────────────────╮
│ URL: https://python.langchain.com/langsmith/python/managed-deep-agents-overview                                 │
│ Content Length: 1,359 characters                                                                                │
│ Preview: **Agent = Model + Harness.**LangChain provides                                                         │
│                                                                                                                 │
│ `create_agent`: a minimal, highly configurable harness. The harness is everything around the model loop: the    │
│ prompt, the tools, and any middleware that shapes behavior. Start with the primitives and compose exactly what  │
│ your use case needs. Supports OpenAI, Anthropic, Google, and more.                                              │
│                                                                                                                 │
│ ## Create an agent                                                                                              │
│                                                                                                                 │
│ This example demonstrates how to create a simple LangChain agent with a custom tool:                            │
│                                                                                                                 │
│ ## Core benefits                                                                                                │
│                                                                                                                 │
│ ## Standard model interface                                                                                     │
│                                                                                                                 │
│ Use one interface for chat models, embeddings, and more across providers. Switch models with minimal code       │
│ changes and keep your application portable as requirements evolve.                                              │
│                                                                                                                 │
│ ## Highly configurable harness                                                                                  │
│                                                                                                                 │
│ Start with                                                                                                      │
│                                                                                                                 │
│ `create_agent` as a minimal harness and add capabilities incrementally through middleware. Compose only what    │
│ your use case needs, from guardrails and retries to routing and custom tool policies.                           │
│                                                                                                                 │
│ ## Built on top of LangGraph                                                                                    │
│                                                                                                                 │
│ LangChain’s agents are built on top of LangGraph. This allows us to...                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🎉 Batch processing complete! Total documents extracted: 9

## 🎯 Real-World Use Cases

Here are some practical applications of TavilyMap and TavilyExtract:

### 1. Documentation Scraping
- Map entire documentation sites
- Extract clean content for search indexes
- Build knowledge bases from existing docs

### 2. Competitive Analysis
- Map competitor websites
- Extract product information
- Monitor content changes

### 3. Research & Content Collection
- Gather information from multiple sources
- Build datasets for analysis
- Create content archives

### 4. SEO & Site Analysis
- Discover all pages on a site
- Analyze content structure
- Identify content gaps


## 🎯 Conclusion

This tutorial demonstrated the power of TavilyMap and TavilyExtract for automated web content discovery and extraction:

### Key Takeaways:

1. **TavilyMap** is perfect for:
   - Discovering website structures
   - Finding all pages on a site
   - Site auditing

2. **TavilyExtract** excels at:
   - Clean content extraction
   - Removing HTML noise
   - Batch processing
   - Structured data collection

3. **Combined** they enable:
   - Complete documentation scraping
   - Automated content pipelines
   - Knowledge base creation
   - Research automation

### Next Steps:
- Integrate with vector databases for semantic search
- Add content filtering and classification
- Build monitoring systems for content changes
- Create automated reporting dashboards

---

**Happy scraping!** 🚀